In [1]:
import arcpy

In [ ]:
candidatos = r"D:\Documents\CityEngine\Default Workspace\Area_estudio\models\estudio.gdb\candidatos_cam_final"     # Feature class de puntos (cada cámara)
observados = r"D:\Documents\ArcGIS\Projects\ProyectoGrado\ProyectoGrado.gdb\puntos_area_estudio_final_limpio"       # Feature class de puntos/vertices a observar
multipatch_obstaculos = r"D:\Documents\ArcGIS\Projects\ProyectoGrado\ProyectoGrado.gdb\Area_estudio_ProceduralMerge"  # Multipatch de edificios
salida_final = r"D:\Documents\ArcGIS\Projects\ProyectoGrado\ProyectoGrado.gdb\All_sample_sight_lines"       # Feature Class final

In [3]:
gdb_temp = r"D:\Documents\UD\2024-3\Proyecto\area_estudio\temp\temp.gdb"

In [4]:
import os
if not arcpy.Exists(salida_final):
    arcpy.CreateFeatureclass_management(os.path.dirname(salida_final),
                                        os.path.basename(salida_final),
                                        "POLYLINE",
                                        spatial_reference=candidatos,
                                        has_z="ENABLED")

In [5]:
arcpy.management.MakeFeatureLayer(observados, "observados")
arcpy.management.MakeFeatureLayer(candidatos, "candidatos")

<Result 'candidatos'>

In [ ]:
cont = 0
with arcpy.da.SearchCursor("candidatos", ["OID@", "SHAPE@"]) as cursor:
    for oid, shape in cursor:
        if(oid) < 17409:
            print(oid)
            continue
        print("PROCESANDO CÁMARA" + str(oid))
        where = f"OBJECTID = {oid}"
        
        if arcpy.Exists("obs_layer"):
            arcpy.management.Delete("obs_layer")
        arcpy.management.MakeFeatureLayer("candidatos", "obs_layer", where)
        # if(cont == 0):
        #     salida_temp = f"D:\Documents\ArcGIS\Projects\ProyectoGrado\ProyectoGrado.gdb\All_sample_sight_lines"
        # else:
        if arcpy.Exists("in_memory/temp_lines"):
            arcpy.Delete_management("in_memory/temp_lines")
        salida_temp = "in_memory/temp_lines"
        
            
        arcpy.management.SelectLayerByLocation(
                    in_layer="observados",
                    overlap_type="INTERSECT_3D",
                    select_features=shape,
                    search_distance="100 Meters",
                    selection_type="NEW_SELECTION",
                    invert_spatial_relationship="NOT_INVERT"
        )
        
        arcpy.ddd.ConstructSightLines("obs_layer", "observados", salida_temp, observer_height_field="Shape.Z", target_height_field="Shape.Z", sample_distance=1, output_the_direction="OUTPUT_THE_DIRECTION", sampling_method="3D_DISTANCE")
        
        #if(cont > 0):
        arcpy.Append_management(salida_temp, "D:\Documents\ArcGIS\Projects\ProyectoGrado\ProyectoGrado.gdb\All_sample_sight_lines", "NO_TEST")

        #cont += 1

    

In [9]:
edm = r"D:\Documents\CityEngine\Default Workspace\Area_estudio\maps\Terrain_Imagery1\elevation.tif"
lineas_vision = r"D:\Documents\ArcGIS\Projects\ProyectoGrado\ProyectoGrado.gdb\lineas_vision_final_completo"
arcpy.ddd.LineOfSight(edm, salida_final, lineas_vision,in_features=multipatch_obstaculos)

<Result 'D:\\Documents\\ArcGIS\\Projects\\ProyectoGrado\\ProyectoGrado.gdb\\lineas_vision_final_completo'>